# Notebook 04 — Probability Calibration and Conformal Prediction Analysis

**Purpose:** Inspect temperature-scaled probability calibration (`NLL`, `Brier`, `ECE`), analyze pooled split-conformal LAC (`k* = 634`, `q* = 0.904267`) versus image-level class-conditional Mondrian LAC (`alpha = 0.10`), distinguish image-level from group-level coverage estimands, examine small-sample `AKIEC` under-coverage, and reproduce LAC prediction-set construction live using `src.modules.selective_decision.conformal.ConformalPredictor` when local logits are supplied.  
**Data & Artifact Requirement:** Aggregate coverage estimands, class-conditional Mondrian tables, and calibration stratum diagnostics execute standalone from tracked CSV/JSON artifacts. Live LAC quantile & set-size recomputation from raw logits requires local `calibration_canonical.npz`, `test_canonical.npz`, and `ISIC_2019_Training_Metadata.csv` (`REQUIRES_DATA` / `REQUIRES_MODEL_ARTIFACT`).

## 1. Learning Objectives & Scientific Context

### Learning Objectives
1. Separate **image-level marginal coverage**, **group-level coverage estimands** (equal-cluster, all-covered, any-covered), **per-class empirical coverage**, and **calibration stratum sample sizes** ($n_c$ images vs. $K_c$ operational groups in $\mathcal{D}_{\mathrm{conf}}$).
2. Understand why **pointwise image-level Wilson confidence intervals** do not account for within-lesion clustering and why observed empirical coverage is **not** an unconditional finite-sample guarantee.
3. Analyze why image-level Mondrian LAC fails to fix `AKIEC` under-coverage (`74.21%` pooled -> `70.59%` Mondrian) due to small calibration counts (`n_c = 30`, `K_c = 17`), and why `DF` (`K_c = 4`) and `VASC` (`K_c = 5`) cannot support cluster-level quantiles.

### Prerequisites
- Core packages (`numpy`, `scipy`, `pandas`, `matplotlib`) and `src.modules.selective_decision.conformal.ConformalPredictor`.

## 2. Environment & Artifact Preflight

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    """Locate repository root portably without hardcoded machine paths."""
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "research" / "derma_act" / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate DERMA-ACT repository root containing pyproject.toml and research/derma_act/config.py")


def find_tables_dir(repo_root: Path) -> Path | None:
    for rel in [
        Path("supplementary") / "tables",
        Path("manuscripts") / "current" / "PAPER_DRAFT_Q2_V3_2_1_PRE_SUBMISSION" / "tables",
    ]:
        cand = repo_root / rel
        if cand.is_dir():
            return cand
    return None


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.modules.selective_decision.conformal import ConformalPredictor

OUT_DIR = REPO_ROOT / "outputs" / "notebooks" / "nb04_conformal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

P71_CSV = REPO_ROOT / "research" / "derma_act" / "p7_1_conformal_quantile_audit" / "P7_1_GROUP_COVERAGE_RECONCILIATION.csv"
PG_SUM_CSV = REPO_ROOT / "research" / "derma_act" / "p8_1_final_submission_audit" / "phase_g_mondrian_conformal" / "phase_g_mondrian_summary.csv"
PG_CLS_CSV = REPO_ROOT / "research" / "derma_act" / "p8_1_final_submission_audit" / "phase_g_mondrian_conformal" / "phase_g_mondrian_per_class.csv"
PG_JSON = REPO_ROOT / "research" / "derma_act" / "p8_1_final_submission_audit" / "phase_g_mondrian_conformal" / "phase_g_feasibility_and_results.json"

CAL_NPZ = REPO_ROOT / "research" / "baseline_v2" / "runs" / "clean_run_001" / "predictions" / "calibration_canonical.npz"
TEST_NPZ = REPO_ROOT / "research" / "baseline_v2" / "runs" / "clean_run_001" / "predictions" / "test_canonical.npz"
META_CSV = REPO_ROOT / "data" / "isic2019" / "raw" / "ISIC_2019_Training_Metadata.csv"

HAS_CONFORMAL_INPUTS = CAL_NPZ.is_file() and TEST_NPZ.is_file() and META_CSV.is_file()
display(pd.DataFrame([
    {"artifact": "P7_1_GROUP_COVERAGE_RECONCILIATION.csv", "available": P71_CSV.is_file(), "tier": "TRACKED"},
    {"artifact": "phase_g_mondrian_{summary,per_class}.csv", "available": PG_SUM_CSV.is_file() and PG_CLS_CSV.is_file(), "tier": "TRACKED"},
    {"artifact": "phase_g_feasibility_and_results.json", "available": PG_JSON.is_file(), "tier": "TRACKED"},
    {"artifact": "calibration/test NPZ + ISIC Metadata CSV", "available": HAS_CONFORMAL_INPUTS, "tier": "REQUIRES_DATA / REQUIRES_MODEL_ARTIFACT"},
]))

## 3. Conformal Estimands & Calibration Subgroup Diagnostics

Split-conformal Least Ambiguous Set-Valued Classification (LAC) defines nonconformity score $s(x, y) = 1 - \hat{p}_T(y \mid x)$ using temperature-calibrated probabilities fit strictly on $\mathcal{D}_{\mathrm{fit}}$ ($N = 698$, $T_{\mathrm{fit}} = 0.855690$). On $\mathcal{D}_{\mathrm{conf}}$ ($N = 703$, $K = 396$ operational groups) at $\alpha = 0.10$:
- **Pooled order statistic:** $k^* = \lceil (703 + 1)(1 - 0.10) \rceil = 634 \implies q^* = 0.904267$ (probability cutoff $1 - q^* = 0.095733$).
- **Image-level Mondrian order statistic:** $k_c = \lceil (n_c + 1)(1 - 0.10) \rceil$ per class $c$. For $n_c = 9$ (`DF`) and $n_c = 10$ (`VASC`), $k_c = n_c$ hits the sample maximum, while $K_c < 9$ makes cluster-level Mondrian quantiles undefined.

In [ ]:
p71_df = pd.read_csv(P71_CSV)
pg_sum = pd.read_csv(PG_SUM_CSV)
pg_cls = pd.read_csv(PG_CLS_CSV)
pg_meta = json.loads(PG_JSON.read_text(encoding="utf-8"))

print("=== Multi-Estimand Pooled LAC Coverage Reconciliation (Image vs. Group Units) ===")
display(p71_df[["estimand_type", "label", "unit", "numerator", "denominator", "empirical_coverage", "cluster_ci95_low", "cluster_ci95_high"]])

print("=== Calibration Subgroup Feasibility Diagnostics (D_conf, N=703, alpha=0.10) ===")
cal_diag = pd.DataFrame(pg_meta["feasibility_audit"]["per_class_calibration_diagnostics"])
display(cal_diag[["class_name", "n_conf_images", "k_conf_clusters", "k_c_image", "hits_exact_sample_max", "cluster_max_feasible", "q_c", "prob_cutoff"]])

print("=== Aggregate Pooled vs. Image-Level Mondrian Summary (Test N=3,686, K=1,972) ===")
display(pg_sum[["method", "marginal_coverage", "coverage_cluster_ci95_low", "coverage_cluster_ci95_high", "equal_cluster_coverage", "mean_set_size", "singleton_rate", "worst_class_coverage"]])

## 4. Live Split-Conformal LAC Reproduction (`REQUIRES_DATA` / `REQUIRES_MODEL_ARTIFACT`)

When local frozen logits and ISIC 2019 metadata are present, this cell reproduces the exact group-stratified $\mathcal{D}_{\mathrm{fit}} / \mathcal{D}_{\mathrm{conf}}$ split (`seed = 20261112`), fits $T_{\mathrm{fit}}$, calibrates `ConformalPredictor(alpha=0.10)` from `src/modules/selective_decision/conformal.py`, and verifies marginal coverage and set-size distribution on Test ($N = 3,686$).

In [ ]:
from scipy.optimize import minimize
from scipy.special import softmax

live_set_sizes = None
if HAS_CONFORMAL_INPUTS:
    meta_df = pd.read_csv(META_CSV).set_index("image")
    cal = np.load(CAL_NPZ, allow_pickle=True)
    tst = np.load(TEST_NPZ, allow_pickle=True)
    cal_ids, y_cal, z_cal = cal["image_id"], cal["true_label"].astype(int), cal["logits"].astype(np.float64)
    y_tst, z_tst = tst["true_label"].astype(int), tst["logits"].astype(np.float64)

    # Operational grouping matching scripts/run_p7_1_conformal_audit.py
    cal_grp = np.array([
        lid if isinstance(lid, str) and lid.strip() else f"IMG::{iid}"
        for iid, lid in ((i, meta_df["lesion_id"].get(i)) for i in cal_ids)
    ])
    ugc = np.unique(cal_grp)
    g_lab = np.array([np.bincount(y_cal[cal_grp == g], minlength=7).argmax() for g in ugc])
    rng = np.random.default_rng(20261112)
    fit_g = []
    for c in range(7):
        gs = ugc[g_lab == c]
        rng.shuffle(gs)
        fit_g.extend(gs[: len(gs) // 2])
    fit_idx = np.where(np.isin(cal_grp, np.array(fit_g)))[0]
    conf_idx = np.where(~np.isin(cal_grp, np.array(fit_g)))[0]

    def fit_nll(log_t):
        t = float(np.exp(log_t[0]))
        p = softmax(z_cal[fit_idx] / t, axis=1)
        return float(-np.log(np.clip(p[np.arange(len(fit_idx)), y_cal[fit_idx]], 1e-12, 1.0)).mean())

    t_fit = float(np.exp(minimize(fit_nll, [0.0], method="BFGS").x[0]))
    p_conf = softmax(z_cal[conf_idx] / t_fit, axis=1)
    p_test = softmax(z_tst / t_fit, axis=1)

    cp = ConformalPredictor(alpha=0.10)
    cp.calibrate(p_conf, y_cal[conf_idx])
    # Exact canonical finite-sample order statistic k* = ceil((n+1)*(1-alpha)) = 634 (0-based index 633) per P7.1
    cal_scores = 1.0 - p_conf[np.arange(len(conf_idx)), y_cal[conf_idx]]
    k_star = int(np.ceil((len(conf_idx) + 1) * (1.0 - 0.10)))
    q_exact = float(np.sort(cal_scores)[k_star - 1])
    exact_sets = [set(np.where(p_test[i] >= 1.0 - q_exact)[0].tolist()) or {int(np.argmax(p_test[i]))} for i in range(len(y_tst))]
    exact_cov = float(np.mean([y_tst[i] in exact_sets[i] for i in range(len(y_tst))]))
    live_set_sizes = np.array([len(s) for s in exact_sets])
    cp_cov = cp.coverage(p_test, y_tst)

    print(f"D_fit={len(fit_idx)}, D_conf={len(conf_idx)}, T_fit={t_fit:.6f}")
    print(f"Canonical P7.1 exact k*={k_star} quantile: q*={q_exact:.6f} -> Test Coverage={100*exact_cov:.2f}% | Mean Set Size={live_set_sizes.mean():.4f} | Singleton={100*(live_set_sizes==1).mean():.2f}%")
    print(f"ConformalPredictor (np.quantile method='higher', rank k*+1): q_hat={cp.q_hat:.6f} -> Test Coverage={100*cp_cov:.2f}% (conservative by 1 order rank)")
else:
    print("SKIPPED (REQUIRES_DATA / REQUIRES_MODEL_ARTIFACT): Raw logits / metadata CSV not present in this clone.")
    print("All summary figures below remain grounded in tracked canonical P7.1 and Phase G CSV ledgers.")

## 5. Visualization: Multi-Estimand Coverage, Per-Class Mondrian Comparison & Set Sizes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

# Panel A: Per-class Pooled vs Mondrian Coverage (with pointwise image-level Wilson CIs)
pooled = pg_cls[pg_cls["method"] == "Pooled_LAC_SplitConformal"].reset_index(drop=True)
mond = pg_cls[pg_cls["method"] == "Mondrian_ClassConditional_LAC"].reset_index(drop=True)
x = np.arange(len(pooled))
w = 0.36
axes[0].bar(x - w / 2, 100 * pooled["coverage"], width=w, label="Pooled LAC (k*=634)", color="#2b6cb0",
            yerr=[100 * (pooled["coverage"] - pooled["wilson_ci95_low"]), 100 * (pooled["wilson_ci95_high"] - pooled["coverage"])], capsize=3)
axes[0].bar(x + w / 2, 100 * mond["coverage"], width=w, label="Image-Level Mondrian LAC", color="#dd6b20",
            yerr=[100 * (mond["coverage"] - mond["wilson_ci95_low"]), 100 * (mond["wilson_ci95_high"] - mond["coverage"])], capsize=3)
axes[0].axhline(90.0, color="black", linestyle="--", linewidth=1.2, label="Nominal 90% Target")
axes[0].set_xticks(x)
xlabels = [f"{r.class_name}\n(n_c={int(cal_diag.loc[i,'n_conf_images'])}, K_c={int(cal_diag.loc[i,'k_conf_clusters'])})" for i, r in pooled.iterrows()]
axes[0].set_xticklabels(xlabels, fontsize=8)
axes[0].set_ylim(55, 106)
axes[0].set_ylabel("Empirical Test Coverage (%) [Image Wilson 95% CI]")
axes[0].set_title("Per-Class Coverage vs. Calibration Stratum Size")
axes[0].legend(fontsize=8, loc="lower right")
axes[0].grid(axis="y", alpha=0.3)

# Panel B: Set-size distribution (live if recomputed, else tracked canonical Pooled LAC counts from tab6)
if live_set_sizes is not None:
    vals, cnts = np.unique(live_set_sizes, return_counts=True)
    panel_b_title = "Pooled LAC Set-Size Distribution (Recomputed Live, N=3,686)"
else:
    vals = np.array([1, 2, 3, 4, 5, 6])
    cnts = np.array([1404, 1409, 654, 185, 33, 1])
    panel_b_title = "Pooled LAC Set-Size Distribution (Tracked Tab 6 Census, N=3,686)"

axes[1].bar(vals, 100.0 * cnts / cnts.sum(), color="#2f855a")
axes[1].set_xticks(vals)
axes[1].set_xlabel("Prediction Set Size |Gamma(x)|")
axes[1].set_ylabel("Share of Held-Out Test Images (%)")
axes[1].set_title(panel_b_title)
axes[1].grid(axis="y", alpha=0.3)
for v, c in zip(vals, cnts):
    axes[1].text(v, 100.0 * c / cnts.sum() + 0.8, f"{c:,}\n({100*c/cnts.sum():.1f}%)", ha="center", fontsize=8)

plt.tight_layout()
fig.savefig(OUT_DIR / "conformal_calibration_analysis.png", dpi=150)
plt.show()

## 6. Interpretation & Critical Statistical Limitations

- **Not an Unconditional Coverage Guarantee:** Finite-sample conformal theory guarantees marginal coverage only under exchangeability. Because multiple photographs belong to the same `lesion_id` cluster (`k >= 2` photos have `88.54%` coverage vs `94.94%` for singletons) and `patient_id` is absent, exchangeability is not verified.
- **Small-Sample `AKIEC` Mondrian Limitation:** `AKIEC` has only `n_c = 30` calibration images across `K_c = 17` clusters in $\mathcal{D}_{\mathrm{conf}}$. Image-level Mondrian calibration degrades `AKIEC` test coverage from `74.21%` to `70.59%` while inflating overall mean set size from `1.925` to `2.390`.
- **Wilson Interval Clustering Caveat:** Pointwise image-level Wilson 95% confidence intervals shown in Panel A treat images as independent Bernoulli trials and therefore **do not account for within-lesion clustering**; always consult the cluster-stratified bootstrap intervals in `P7_1_GROUP_COVERAGE_RECONCILIATION.csv` for inference.

### Reproduction & References
- Deterministic split & bootstrap seed: `20261112`
- Output saved to: `outputs/notebooks/nb04_conformal/conformal_calibration_analysis.png`
- Related scripts/docs: [`scripts/run_p7_1_conformal_audit.py`](../scripts/run_p7_1_conformal_audit.py) · [`scripts/run_phase_g_mondrian_conformal.py`](../scripts/run_phase_g_mondrian_conformal.py) · [`src/modules/selective_decision/conformal.py`](../src/modules/selective_decision/conformal.py)